In [1]:
import os
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
from shapely.geometry import Point
from geopy.geocoders import Nominatim
import warnings
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

# Geocoder initialization
geolocator= Nominatim(user_agent="cmpt353_jupyter_routing", timeout=10)
print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Set base directory (Adjust if your notebook is located elsewhere)
base_dir = os.path.dirname(os.path.dirname(os.getcwd()))

print("1. Loading Street Network")
graph_path = os.path.join(base_dir, "Data", "Processed", "burnaby_walk.graphml")
if os.path.exists(graph_path):
    G_proj = ox.load_graphml(graph_path)
else:
    G = ox.graph_from_place("Burnaby, British Columbia, Canada", network_type='walk')
    G_proj = ox.project_graph(G, to_crs='EPSG:26910')
    ox.save_graphml(G_proj, graph_path)

print("2. Loading Crime Data")
crime_path = os.path.join(base_dir, "Data", "Processed", "harmonized_crime_data.geojson")
crimes = gpd.read_file(crime_path)
if crimes.crs != 'EPSG:26910':
    crimes = crimes.to_crs('EPSG:26910')

print("3. Loading Transit Data (GTFS)")
transit_dir = os.path.join(base_dir, "Data", "Transit Data")
stops = pd.read_csv(os.path.join(transit_dir, "stops.txt"), dtype={'stop_id': str})
stop_times = pd.read_csv(os.path.join(transit_dir, "stop_times.txt"), dtype={'stop_id': str, 'trip_id': str})
trips = pd.read_csv(os.path.join(transit_dir, "trips.txt"), dtype={'trip_id': str, 'route_id': str, 'shape_id': str})
routes = pd.read_csv(os.path.join(transit_dir, "routes.txt"), dtype={'route_id': str, 'route_short_name': str})
shapes = pd.read_csv(os.path.join(transit_dir, "shapes.txt"), dtype={'shape_id': str})

#  EXCLUDE NIGHT BUSES 

print("Filtering out Night Buses")
routes = routes[~routes['route_short_name'].str.match(r'^N\d+', na=False)]

trips = trips[trips['route_id'].isin(routes['route_id'])]
stop_times = stop_times[stop_times['trip_id'].isin(trips['trip_id'])]

# Map Stop to Route short names
st_tr = pd.merge(stop_times[['stop_id', 'trip_id']], trips[['trip_id', 'route_id']], on='trip_id')
st_tr_rt = pd.merge(st_tr, routes[['route_id', 'route_short_name']], on='route_id')
stop_routes = st_tr_rt.groupby('stop_id')['route_short_name'].unique().apply(lambda x: ", ".join(map(str, x))).reset_index()
stops_merged = pd.merge(stops, stop_routes, on='stop_id', how='left')

stops_gdf = gpd.GeoDataFrame(
    stops_merged, 
    geometry=gpd.points_from_xy(stops_merged.stop_lon, stops_merged.stop_lat),
    crs="EPSG:4326"
).to_crs("EPSG:26910")

# Build Transit Graph for Transfer routing
stop_times['stop_sequence'] = stop_times['stop_sequence'].astype(int)
st_sorted = stop_times.sort_values(by=['trip_id', 'stop_sequence'])
st_sorted['next_stop_id'] = st_sorted.groupby('trip_id')['stop_id'].shift(-1)
edges_df = st_sorted.dropna(subset=['next_stop_id'])[['stop_id', 'next_stop_id']].drop_duplicates()

edges_df['weight'] = 1 
G_transit = nx.from_pandas_edgelist(edges_df, 'stop_id', 'next_stop_id', edge_attr='weight', create_using=nx.Graph())

# Allow transfers between stops with the same name (Penalty weight = 10)
for name, group in stops.groupby('stop_name'):
    s_ids = group['stop_id'].tolist()
    for i in range(len(s_ids)):
        for j in range(i+1, len(s_ids)):
            G_transit.add_edge(s_ids[i], s_ids[j], weight=10)

print("All data loaded successfully!")

1. Loading Street Network...
2. Loading Crime Data...
3. Loading Transit Data (GTFS)...
Filtering out Night Buses...
All data loaded successfully!


In [ ]:
def smart_geocode(query):
    res = geolocator.geocode(query, exactly_one=True, country_codes='ca')
    if not res and "," not in query:
        res = geolocator.geocode(f"{query}, British Columbia, Canada", exactly_one=True, country_codes='ca')
    if res:
        print(f"Found Address: {res.address}")
        return (res.latitude, res.longitude)
    return None

def get_transit_route(start_geom, end_geom):
    """
    Finds the optimal transit path including transfers, 
    and pieces together the exact curved shapes for every bus/train taken.
    """
    if start_geom.distance(end_geom) < 1200:
        return None, None, None, None, None
        
    stops_gdf['dist_to_orig'] = stops_gdf.distance(start_geom)
    stops_gdf['dist_to_dest'] = stops_gdf.distance(end_geom)
    
    start_cands = stops_gdf.nsmallest(5, 'dist_to_orig')
    end_cands = stops_gdf.nsmallest(5, 'dist_to_dest')
    
    best_path = None
    best_cost = float('inf')
    start_stop, end_stop = None, None
    

    for _, s in start_cands.iterrows():
        for _, e in end_cands.iterrows():
            try:
                path = nx.shortest_path(G_transit, s['stop_id'], e['stop_id'], weight='weight')
                cost = nx.shortest_path_length(G_transit, s['stop_id'], e['stop_id'], weight='weight')
                
                total_dist = s['dist_to_orig'] + e['dist_to_dest'] + (cost * 200) 
                if total_dist < best_cost:
                    best_cost = total_dist
                    best_path = path
                    start_stop = s
                    end_stop = e
            except nx.NetworkXNoPath:
                continue
                
    if not best_path:
        return None, None, None, None, None
        
    shape_coords = []
    route_names = []

    for i in range(len(best_path)-1):
        s1, s2 = best_path[i], best_path[i+1]
        
        if G_transit.edges[s1, s2]['weight'] == 10:
            s1_geom = stops_gdf[stops_gdf['stop_id'] == s1].iloc[0]
            s2_geom = stops_gdf[stops_gdf['stop_id'] == s2].iloc[0]
            shape_coords.extend([(s1_geom['stop_lat'], s1_geom['stop_lon']), (s2_geom['stop_lat'], s2_geom['stop_lon'])])
            continue
            
        st1 = stop_times[stop_times['stop_id'] == s1]
        st2 = stop_times[stop_times['stop_id'] == s2]
        merged = pd.merge(st1, st2, on='trip_id', suffixes=('_1', '_2'))
        valid = merged[merged['stop_sequence_1'] < merged['stop_sequence_2']]
        
        if not valid.empty:
            trip_id = valid.iloc[0]['trip_id']
            trip_info = trips[trips['trip_id'] == trip_id].iloc[0]
            
   
            r_id = trip_info['route_id']
            r_name = routes[routes['route_id'] == r_id].iloc[0].get('route_short_name', str(r_id))
            if r_name not in route_names:
                route_names.append(r_name)
                
            shape_id = trip_info['shape_id']
            if pd.notna(shape_id):
                t_shape = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')
                s1_lat, s1_lon = stops_gdf[stops_gdf['stop_id']==s1].iloc[0][['stop_lat', 'stop_lon']]
                s2_lat, s2_lon = stops_gdf[stops_gdf['stop_id']==s2].iloc[0][['stop_lat', 'stop_lon']]
                
                t_shape['d1'] = (t_shape['shape_pt_lat'] - s1_lat)**2 + (t_shape['shape_pt_lon'] - s1_lon)**2
                t_shape['d2'] = (t_shape['shape_pt_lat'] - s2_lat)**2 + (t_shape['shape_pt_lon'] - s2_lon)**2
                
                seq1 = t_shape.loc[t_shape['d1'].idxmin(), 'shape_pt_sequence']
                seq2 = t_shape.loc[t_shape['d2'].idxmin(), 'shape_pt_sequence']
                
                if seq1 <= seq2:
                    trimmed = t_shape[(t_shape['shape_pt_sequence'] >= seq1) & (t_shape['shape_pt_sequence'] <= seq2)]
                else:
                    trimmed = t_shape[(t_shape['shape_pt_sequence'] <= seq1) & (t_shape['shape_pt_sequence'] >= seq2)].sort_values('shape_pt_sequence', ascending=False)
                    
                shape_coords.extend(list(zip(trimmed['shape_pt_lat'], trimmed['shape_pt_lon'])))
            else:
                s1_geom = stops_gdf[stops_gdf['stop_id'] == s1].iloc[0]
                s2_geom = stops_gdf[stops_gdf['stop_id'] == s2].iloc[0]
                shape_coords.extend([(s1_geom['stop_lat'], s1_geom['stop_lon']), (s2_geom['stop_lat'], s2_geom['stop_lon'])])

    final_route_name = " ➔ ".join(route_names) if route_names else "Transit"
    
    return shape_coords, start_stop, end_stop, final_route_name, best_path

def compute_fast_cost_and_routes(start_geom, end_geom):
    buffer_dist = 1500 
    minx, maxx = min(start_geom.x, end_geom.x) - buffer_dist, max(start_geom.x, end_geom.x) + buffer_dist
    miny, maxy = min(start_geom.y, end_geom.y) - buffer_dist, max(start_geom.y, end_geom.y) + buffer_dist
    
    valid_nodes = [n for n, d in G_proj.nodes(data=True) if minx <= d['x'] <= maxx and miny <= d['y'] <= maxy]
    if not valid_nodes: return [], [], None
        
    G_local = G_proj.subgraph(valid_nodes).copy()
    _, edges_local = ox.graph_to_gdfs(G_local)
    crimes_local = crimes.cx[minx:maxx, miny:maxy]
    
    edges_buffered = edges_local.copy()
    edges_buffered['geometry'] = edges_buffered.geometry.buffer(50)
    edges_buffered = edges_buffered.reset_index()
    
    joined = gpd.sjoin(crimes_local, edges_buffered, how='inner', predicate='within')
    crime_counts = joined.groupby(['u', 'v', 'key']).size().reset_index(name='crime_count')
    
    edges_local = edges_local.reset_index().merge(crime_counts, on=['u', 'v', 'key'], how='left')
    edges_local['crime_count'] = edges_local['crime_count'].fillna(0)
    edges_local['crime_density'] = edges_local['crime_count'] / edges_local['length']
    edges_local = edges_local.set_index(['u', 'v', 'key'])
    
    min_len, max_len = edges_local['length'].min(), edges_local['length'].max()
    min_crime, max_crime = edges_local['crime_density'].min(), edges_local['crime_density'].max()
    
    def min_max_scale(val, min_v, max_v): return 0.0 if max_v == min_v else (val - min_v) / (max_v - min_v)
    
    for u, v, key, data in G_local.edges(keys=True, data=True):
        ds = edges_local.loc[(u, v, key), 'crime_density']
        density = ds.iloc[0] if isinstance(ds, pd.Series) else ds
        n_len = min_max_scale(data.get('length', 1.0), min_len, max_len)
        n_crime = min_max_scale(density, min_crime, max_crime)
        data['cost_shortest'] = n_len
        data['cost_safest'] = (0.3 * n_len) + (0.7 * n_crime) 

    orig_node = ox.nearest_nodes(G_local, start_geom.x, start_geom.y)
    dest_node = ox.nearest_nodes(G_local, end_geom.x, end_geom.y)
    if orig_node == dest_node: return [orig_node], [orig_node], G_local
        
    r_short = nx.shortest_path(G_local, orig_node, dest_node, weight='cost_shortest')
    r_safe = nx.shortest_path(G_local, orig_node, dest_node, weight='cost_safest')
    return r_short, r_safe, G_local

print("Functions ready.")

✅ Functions ready.


In [4]:
from IPython.display import display, HTML

# 1. Define Input Addresses Here
origin_query = "Metropolis at Metrotown, Burnaby"
dest_query = "7710 Kentwood street, Burnaby"

print("Geocoding locations")
origin_latlon = smart_geocode(origin_query)
dest_latlon = smart_geocode(dest_query)

if origin_latlon and dest_latlon:
    orig_geom = gpd.GeoSeries([Point(origin_latlon[1], origin_latlon[0])], crs='EPSG:4326').to_crs('EPSG:26910').iloc[0]
    dest_geom = gpd.GeoSeries([Point(dest_latlon[1], dest_latlon[0])], crs='EPSG:4326').to_crs('EPSG:26910').iloc[0]
    
    print("Calculating Transit Route")
    transit_coords, start_stop, end_stop, route_name, transit_path_nodes = get_transit_route(orig_geom, dest_geom)
    using_transit = transit_coords is not None
    
    fm_short, fm_safe, G_map_fm = [], [], None
    lm_short, lm_safe, G_map_lm = [], [], None
    
    if using_transit:
        print("Analyzing First Mile (Origin to Station)")
        fm_short, fm_safe, G_fm = compute_fast_cost_and_routes(orig_geom, start_stop.geometry)
        if G_fm: G_map_fm = ox.project_graph(G_fm, to_crs='EPSG:4326')
            
        print("Analyzing Last Mile (Station to Destination)")
        lm_short, lm_safe, G_lm = compute_fast_cost_and_routes(end_stop.geometry, dest_geom)
        if G_lm: G_map_lm = ox.project_graph(G_lm, to_crs='EPSG:4326')
    else:
        print("Locations are close. Computing Direct Walk")
        fm_short, fm_safe, G_fm = compute_fast_cost_and_routes(orig_geom, dest_geom)
        if G_fm: G_map_fm = ox.project_graph(G_fm, to_crs='EPSG:4326')

    # 2. Build the Map
    print("Generating Map")
    mid_lat = (origin_latlon[0] + dest_latlon[0]) / 2
    mid_lon = (origin_latlon[1] + dest_latlon[1]) / 2
    
    m = folium.Map(location=[mid_lat, mid_lon], zoom_start=13, tiles='OpenStreetMap')
    
    def get_coords(route, graph_map):
        return [(graph_map.nodes[n]['y'], graph_map.nodes[n]['x']) for n in route]
        
    # Draw Walking Routes
    if G_map_fm is not None and len(fm_short) >= 2:
        end_pt = (start_stop['stop_lat'], start_stop['stop_lon']) if using_transit else dest_latlon
        short_c = [origin_latlon] + get_coords(fm_short, G_map_fm) + [end_pt]
        safe_c = [origin_latlon] + get_coords(fm_safe, G_map_fm) + [end_pt]
        folium.PolyLine(short_c, color='red', weight=4, opacity=0.7, popup='First Mile: Shortest').add_to(m)
        folium.PolyLine(safe_c, color='green', weight=6, opacity=0.9, popup='First Mile: Safest').add_to(m)
        
    if using_transit and G_map_lm is not None and len(lm_short) >= 2:
        short_c = [(end_stop['stop_lat'], end_stop['stop_lon'])] + get_coords(lm_short, G_map_lm) + [dest_latlon]
        safe_c = [(end_stop['stop_lat'], end_stop['stop_lon'])] + get_coords(lm_safe, G_map_lm) + [dest_latlon]
        folium.PolyLine(short_c, color='red', weight=4, opacity=0.7, popup='Last Mile: Shortest').add_to(m)
        folium.PolyLine(safe_c, color='green', weight=6, opacity=0.9, popup='Last Mile: Safest').add_to(m)
        
  
    if using_transit:
        folium.PolyLine(transit_coords, color='#00A8E1', weight=6, opacity=1.0, tooltip=f'Transit Route: {route_name}').add_to(m)
        
        for i, s_id in enumerate(transit_path_nodes):
            s_info = stops_merged[stops_merged['stop_id'] == s_id].iloc[0]
            lat, lon = float(s_info['stop_lat']), float(s_info['stop_lon'])
            
            stop_name = s_info.get('stop_name', 'Stop')
            
            if i == 0:
                first_route = route_name.split(' ➔ ')[0]
                folium.Marker((lat, lon), tooltip=f"<b>Board: {first_route}</b>", icon=folium.Icon(color='blue', icon='bus', prefix='fa')).add_to(m)
            elif i == len(transit_path_nodes) - 1:
                folium.Marker((lat, lon), tooltip="<b>Alight Here</b>", icon=folium.Icon(color='blue', icon='flag', prefix='fa')).add_to(m)
            else:
                folium.CircleMarker(
                    location=(lat, lon),
                    radius=5, color='white', weight=2, fill=True, fill_color='#00A8E1', fill_opacity=1.0,
                    tooltip=f"<b>Transfer / Stop:</b> {stop_name}"
                ).add_to(m)
                
    # Draw Origin/Dest
    folium.Marker(origin_latlon, tooltip="Origin", icon=folium.Icon(color='orange', icon='home')).add_to(m)
    folium.Marker(dest_latlon, tooltip="Destination", icon=folium.Icon(color='red', icon='star')).add_to(m)
    
    # Draw Crimes (Filtering locally)
    crimes_4326 = crimes.to_crs('EPSG:4326')
    buffer_deg = 0.005 # Roughly 500m in coordinates
    
    crimes_orig = crimes_4326.cx[origin_latlon[1]-buffer_deg:origin_latlon[1]+buffer_deg, 
                                 origin_latlon[0]-buffer_deg:origin_latlon[0]+buffer_deg]
    crimes_dest = crimes_4326.cx[dest_latlon[1]-buffer_deg:dest_latlon[1]+buffer_deg, 
                                 dest_latlon[0]-buffer_deg:dest_latlon[0]+buffer_deg]
    
    crimes_to_plot = pd.concat([crimes_orig, crimes_dest]).drop_duplicates()
    
    crime_group = folium.FeatureGroup(name="Crime Incidents")
    for _, row in crimes_to_plot.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=3, color='crimson', fill=True, fill_opacity=0.5,
            popup=row.get('Crime_Category', 'Crime')
        ).add_to(crime_group)
        
    crime_group.add_to(m)
    folium.LayerControl().add_to(m)
    
    print("Done! Displaying map")
    display(HTML(m._repr_html_()))
    
else:
    print("Could not find valid coordinates for the given addresses.")

Geocoding locations
Found Address: Metropolis at Metrotown, 4700, Kingsway, Metrotown, Burnaby, Metro Vancouver Regional District, British Columbia, V5H 4N2, Canada
Found Address: 7710, Kentwood Street, Burnaby, Metro Vancouver Regional District, British Columbia, V5A 2H4, Canada
Calculating Transit Route
Analyzing First Mile (Origin to Station)
Analyzing Last Mile (Station to Destination)
Generating Map
Done! Displaying map
